In [4]:
%pip install sqlalchemy pymysql

  Using cached pymysql-1.2.4-py3-none-any.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.4 MB 3.7 MB/s eta 0:00:01
   -------------------------------------- - 2.4/2.4 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.8 MB/s eta 0:00:00
Using cached pymysql-1.2.4-py3-none-any.whl (46 kB)
Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\python311\lib\site-packages\vboxapi-1.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

accounts = pd.read_csv("data/raw/ravenstack_accounts.csv")
subs = pd.read_csv("data/raw/ravenstack_subscriptions.csv")
usage = pd.read_csv("data/raw/ravenstack_feature_usage.csv")
tickets = pd.read_csv("data/raw/ravenstack_support_tickets.csv")
churn = pd.read_csv("data/raw/ravenstack_churn_events.csv")

print("Files loaded! Accounts rows:", len(accounts))

Files loaded! Accounts rows: 500


In [13]:
tables = {
    "accounts": accounts,
    "subscriptions": subs,
    "feature_usage": usage,
    "support_tickets": tickets,
    "churn_events": churn
}

print("--- DATA QUALITY AUDIT ---") 

for name,df in tables.items():
    dupes=df.duplicated().sum()

    nulls={}
    for col,count in df.isnull().sum().items():
        if count>0:
            nulls[col]=count

    print(f"[{name.upper()}] Rows:{len(df)} | Dupes: {dupes} | Cols: {list(df.columns)}")
    if nulls:
        print(f"   Nulls:{nulls}")


--- DATA QUALITY AUDIT ---
[ACCOUNTS] Rows:500 | Dupes: 0 | Cols: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
[SUBSCRIPTIONS] Rows:5000 | Dupes: 0 | Cols: ['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']
   Nulls:{'end_date': 4514}
[FEATURE_USAGE] Rows:25000 | Dupes: 0 | Cols: ['usage_id', 'subscription_id', 'usage_date', 'feature_name', 'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature']
[SUPPORT_TICKETS] Rows:2000 | Dupes: 0 | Cols: ['ticket_id', 'account_id', 'submitted_at', 'closed_at', 'resolution_time_hours', 'priority', 'first_response_time_minutes', 'satisfaction_score', 'escalation_flag']
   Nulls:{'satisfaction_score': 825}
[CHURN_EVENTS] Rows:600 | Dupes: 0 | Cols: ['churn_event_id', 'account_id', 'churn_da

In [16]:
print("--- PRIMARY KEY CHECK ---")
for name,df in tables.items():
    pk_cols=[]
    for col in df.columns:
        if df[col].nunique()==len(df) and df[col].isnull().sum()==0:
             pk_cols.append(col)
    print(f"[{name.upper()}] Primary Key candidates: {pk_cols}")

--- PRIMARY KEY CHECK ---
[ACCOUNTS] Primary Key candidates: ['account_id', 'account_name']
[SUBSCRIPTIONS] Primary Key candidates: ['subscription_id']
[FEATURE_USAGE] Primary Key candidates: []
[SUPPORT_TICKETS] Primary Key candidates: ['ticket_id']
[CHURN_EVENTS] Primary Key candidates: ['churn_event_id']


In [18]:
print("Usage columns:",list(usage.columns))
print("Total rows:", len(usage), "| Unique usage_id:", usage[usage.columns[0]].nunique())

Usage columns: ['usage_id', 'subscription_id', 'usage_date', 'feature_name', 'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature']
Total rows: 25000 | Unique usage_id: 24979


In [19]:
accounts=accounts.drop_duplicates(subset=["account_id"])
subs = subs.drop_duplicates(subset=["subscription_id"])
usage = usage.drop_duplicates(subset=["usage_id"])
tickets = tickets.drop_duplicates(subset=["ticket_id"])
churn = churn.drop_duplicates(subset=["churn_event_id"])

In [20]:
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"], errors="coerce")
subs["start_date"] = pd.to_datetime(subs["start_date"], errors="coerce")
subs["end_date"] = pd.to_datetime(subs["end_date"], errors="coerce")
usage["usage_date"] = pd.to_datetime(usage["usage_date"], errors="coerce")
churn["churn_date"] = pd.to_datetime(churn["churn_date"], errors="coerce")

C:\tmp\ipykernel_16628\1624133312.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  usage["usage_date"] = pd.to_datetime(usage["usage_date"], errors="coerce")
